# Module 3 — Topic 1: The EDA Process
## Live Demo Notebook

**Scenario:** You've been handed a raw export of 500 Jumia Nigeria sales records.
Your manager says: *"Explore this — tell me what's interesting before we do anything with it."*

Before writing a single line of code, we write our questions.

---

**My EDA questions:**
1. Which product category generates the most revenue?
2. Is there a relationship between product rating and units sold?
3. Are there any suspiciously large or small order values?

---

## Part 1 — Load

In [ ]:
import pandas as pd

df = pd.read_csv('jumia_sales.csv')

# How big is this dataset?
print('Shape:', df.shape)

# What columns do we have?
print('\nColumns:')
print(df.columns.tolist())

In [ ]:
# First look at the data
df.head()

---
## Part 2 — Inspect

In [ ]:
# Data types and missing value counts
df.info()

In [ ]:
# Statistical summary of all numeric columns
df.describe()

In [ ]:
# Let's highlight the mean vs median gap for revenue
mean_rev   = df['revenue'].mean()
median_rev = df['revenue'].median()

print(f'Revenue mean:   NGN {mean_rev:,.0f}')
print(f'Revenue median: NGN {median_rev:,.0f}')
print(f'Difference:     NGN {mean_rev - median_rev:,.0f}')
print()
print('>>> The mean is much higher than the median.')
print('>>> This strongly suggests outliers are pulling the mean up.')
print('>>> We will investigate this properly in Topic 2.')

In [ ]:
# How many orders per category?
print('Orders per category:')
print(df['category'].value_counts())
print()
print('As proportions:')
print(df['category'].value_counts(normalize=True).round(2))

In [ ]:
# Delivery status breakdown
print('Delivery status counts:')
print(df['delivery_status'].value_counts())
print()
print('Payment method counts:')
print(df['payment_method'].value_counts())

---
## Part 3 — Clean

Before exploring, we quickly check for data quality issues.

In [ ]:
# Check for missing values
print('Missing values per column:')
print(df.isnull().sum())
print()

# Check for duplicates
print('Duplicate rows:', df.duplicated().sum())
print()
print('>>> This dataset is clean — no missing values, no duplicates.')
print('>>> We can move straight to exploration.')

---
## Part 4 — Explore

Now we answer our three EDA questions one by one.

In [ ]:
# Question 1: Which product category generates the most total revenue?
revenue_by_category = (
    df.groupby('category')['revenue']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
revenue_by_category.columns = ['category', 'total_revenue']
revenue_by_category['total_revenue_formatted'] = revenue_by_category['total_revenue'].apply(lambda x: f'NGN {x:,.0f}')
print(revenue_by_category[['category', 'total_revenue_formatted']].to_string(index=False))

In [ ]:
# Follow-up question: What about average revenue per order by category?
# (Total revenue can be high just because there are more orders)
avg_revenue = (
    df.groupby('category')['revenue']
    .mean()
    .sort_values(ascending=False)
    .round(0)
)
print('Average revenue per order by category:')
for cat, avg in avg_revenue.items():
    print(f'  {cat:<25} NGN {avg:,.0f}')

In [ ]:
# Question 2: Is there a relationship between product rating and units sold?
correlation = df['rating'].corr(df['units_sold'])
print(f'Correlation between rating and units_sold: {correlation:.3f}')
print()
if abs(correlation) < 0.2:
    print('>>> Weak correlation — rating alone does not strongly predict units sold.')
elif abs(correlation) < 0.5:
    print('>>> Moderate correlation — some relationship exists.')
else:
    print('>>> Strong correlation — rating is a significant predictor of units sold.')

In [ ]:
# Question 3: Are there any suspiciously large or small order values?
print('Revenue distribution summary:')
print(df['revenue'].describe())
print()

# Flag orders above NGN 500,000 as potentially anomalous
high_value = df[df['revenue'] > 500000]
print(f'Orders above NGN 500,000: {len(high_value)}')
if len(high_value) > 0:
    print(high_value[['order_id', 'category', 'product_name', 'quantity', 'unit_price', 'revenue']])

In [ ]:
# Bonus exploration: Which states are ordering the most?
print('Top 5 states by order count:')
print(df['state'].value_counts().head(5))
print()

print('Top 5 states by total revenue:')
state_revenue = df.groupby('state')['revenue'].sum().sort_values(ascending=False).head(5)
for state, rev in state_revenue.items():
    print(f'  {state:<15} NGN {rev:,.0f}')

---
## Part 5 — Hypothesise

Based on what we found, here are our working hypotheses:

| Observation | Hypothesis |
|---|---|
| Phones & Tablets / Electronics dominate revenue | *High-value categories — restocking priority should reflect this* |
| Revenue mean >> median | *A small number of extreme orders are inflating the average — investigate as outliers in Topic 2* |
| Weak correlation between rating and units_sold | *Factors other than rating (price, category, state) likely drive purchase volume* |
| Lagos and Abuja lead on both order count and revenue | *Other states may be underserved — potential growth markets* |
| Orders above NGN 500k exist | *These may be bulk or wholesale orders — should be tracked separately* |

---
## Summary

In this demo we:
- Wrote our EDA questions **before** running any code
- **Loaded** the dataset and checked its shape and columns
- **Inspected** with `info()`, `describe()`, and `value_counts()`
- Spotted a key signal: revenue mean >> median (outlier warning)
- **Cleaned** — confirmed no issues in this dataset
- **Explored** all three EDA questions using groupby and correlation
- **Hypothesised** — documented findings as testable statements

**Next — Topic 2:** We go deeper on the outliers we spotted, and learn how to formally detect patterns and anomalies using IQR, z-scores, and skewness analysis.